In [ ]:
storage_key = dbutils.secrets.get(
    scope="your-secret-scope",
    key="azure-storage-key"
)

spark.conf.set(
    "fs.azure.account.key.stgrocerywaste.dfs.core.windows.net",
    storage_key
)

In [ ]:
from pyspark.sql.functions import round

In [ ]:
gold_path   = "abfss://gold@stgrocerywaste.dfs.core.windows.net/analytics"

In [ ]:
from pyspark.sql.functions import col, sum as spark_sum, count, avg, when

df = spark.read.format("delta").load(
    "abfss://silver@stgrocerywaste.dfs.core.windows.net/inventory"
)

In [ ]:
#STORE WASTE COST TABLE
store_waste = df.withColumn(
    "Waste_Cost",
    col("Unsold_Quantity") * col("Unit_Cost")
).groupBy("Store_ID") \
 .agg(
     round(spark_sum("Waste_Cost"),2).alias("Total_Waste_Cost"),
     round(spark_sum("Unsold_Quantity"),2).alias("Total_Unsold")
 ) \
 .orderBy(col("Total_Waste_Cost").desc())
store_waste.show(10, truncate=False)

In [ ]:
category_waste = df.withColumn(
    "Waste_Cost",
    col("Unsold_Quantity") * col("Unit_Cost")
).groupBy("Product_Category") \
 .agg(
     round(spark_sum("Waste_Cost"), 2).alias("Total_Waste_Cost"),
     count("*").alias("Total_Records")
 ) \
 .orderBy(col("Total_Waste_Cost").desc())

category_waste.show(10, truncate=False)

In [ ]:
#TOP WASTED PRODUCTS
product_waste = df.withColumn(
    "Waste_Cost",
    col("Unsold_Quantity") * col("Unit_Cost")
).groupBy("Product_Name") \
 .agg(
     spark_sum("Waste_Cost").alias("Total_Waste_Cost")
 ) \
 .orderBy(col("Total_Waste_Cost").desc())
product_waste.show(10, truncate=False)

In [ ]:
from pyspark.sql.functions import col, when, sum as spark_sum, count

expiry_loss = df.withColumn(
    "Is_Expired",
    when(col("Expiry_Date") < col("Scan_Date"), 1).otherwise(0)
).groupBy("Product_Name") \
 .agg(
     round(spark_sum("Is_Expired"),2).alias("Expired_Count"),
     count("*").alias("Total_Batches")
 )
expiry_loss.show(10, truncate=False)

In [ ]:
#DAILY WASTE TREND
daily_trend = df.withColumn(
    "Waste_Cost",
    col("Unsold_Quantity") * col("Unit_Cost")
).groupBy("Scan_Date") \
 .agg(
     round(spark_sum("Waste_Cost"),2).alias("Daily_Waste_Cost")
 )
daily_trend.show(10, truncate=False)

In [ ]:
#STORE × CATEGORY ANALYSIS
store_category = df.withColumn(
    "Waste_Cost",
    col("Unsold_Quantity") * col("Unit_Cost")
).groupBy("Store_ID", "Product_Category") \
 .agg(
    round(spark_sum("Waste_Cost"),2).alias("Waste_Cost")
 )
store_category.show(10, truncate=False)

In [ ]:
from pyspark.sql.functions import col, datediff, count

high_risk = df.withColumn(
    "Days_To_Expire",
    datediff(col("Expiry_Date"), col("Scan_Date"))
).filter(col("Days_To_Expire") <= 3) \
 .groupBy("Product_Name") \
 .agg(
     count("*").alias("High_Risk_Count")
 ) \
 .orderBy(col("High_Risk_Count").desc())

high_risk.show(10, truncate=False)
 

In [ ]:
#DISCOUNT vs WASTE IMPACT
discount_impact = df.groupBy("Discount_Applied") \
 .agg(
     round(spark_sum(col("Unsold_Quantity")), 2).alias("Total_Unsold"),
     count("*").alias("Transactions")
 )

discount_impact.show(10, truncate=False)

In [ ]:
store_waste.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save(gold_path + "/store_waste")

category_waste.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save(gold_path + "/category_waste")

product_waste.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save(gold_path + "/product_waste")

expiry_loss.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save(gold_path + "/expiry_loss")

daily_trend.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save(gold_path + "/daily_trend")

store_category.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save(gold_path + "/store_category")

high_risk.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save(gold_path + "/high_risk")

discount_impact.write.mode("overwrite").option("overwriteSchema", "true").format("delta").save(gold_path + "/discount_impact")

In [ ]:
gold_path = "abfss://gold@stgrocerywaste.dfs.core.windows.net/analytics"

store_waste.write.mode("overwrite").format("delta").save(gold_path + "/store_waste")
category_waste.write.mode("overwrite").format("delta").save(gold_path + "/category_waste")
product_waste.write.mode("overwrite").format("delta").save(gold_path + "/product_waste")
discount_impact.write.mode("overwrite").format("delta").save(gold_path + "/discount_impact")
high_risk.write.mode("overwrite").format("delta").save(gold_path + "/high_risk")